In [ ]:
tables = spark.sql("SHOW TABLES IN silver").collect()

matches = []

for row in tables:
    table_name = row["tableName"]
    full_name = f"silver.{table_name}"
    try:
        cols = spark.sql(f"DESCRIBE {full_name}").collect()
        for c in cols:
            col_name = c["col_name"]
            if col_name:
                cname = col_name.strip().lower()
                if "cprod" in cname or "group_conformed" in cname:
                    matches.append((full_name, col_name))
    except Exception:
        pass

display(spark.createDataFrame(matches, ["table_name", "column_name"]))

In [ ]:
SELECT TOP 100
    service_id,
    service_src_id,
    service_src_name,
    service_src_sys_inst_id,
    service_name_conformed,
    z_src_sys_inst_id
FROM silver_rdm_service
WHERE service_src_sys_inst_id = 'MPB001'
ORDER BY service_src_name;

In [ ]:
SELECT TOP 100
    service_id,
    service_src_id,
    service_src_name,
    service_src_sys_inst_id,
    service_name_conformed,
    z_src_sys_inst_id
FROM silver_rdm_service
WHERE service_src_sys_inst_id LIKE 'SONE%'
ORDER BY service_src_sys_inst_id, service_src_name;

In [ ]:
%%sql

SELECT DISTINCT
    serv.value as mpb_service_value
FROM
(
    SELECT user_id,title,updated_at,value,
           ROW_NUMBER() OVER (PARTITION BY user_id ORDER BY updated_at DESC) as rowno
    FROM silver_drj_anamnesises_epicrisis
    WHERE title = 'Treatment Type'
) serv
WHERE serv.rowno = 1
ORDER BY serv.value;

: 

In [ ]:
%%sql

SELECT DISTINCT
    value as mpb_referral_source_value
FROM
(
    SELECT
        id,
        name,
        value
    FROM silver_drj_users_restinfo
    WHERE name = 'Referral Source'
) src
ORDER BY mpb_referral_source_value;

In [ ]:
%%sql

SELECT DISTINCT
    contr_mstr_service_conformed
FROM silver_contract
WHERE contr_id LIKE 'MPB%';

prod check

In [ ]:
%%sql

SELECT
    service_id,
    service_src_id,
    service_src_name,
    service_src_sys_inst_id,
    service_name_conformed,
    z_src_sys_inst_id
FROM silver_rdm_service
WHERE service_src_sys_inst_id = 'MPB001'
ORDER BY service_src_name;

In [ ]:
%%sql

SELECT DISTINCT
    value as mpb_treatment_type_value
FROM
(
    SELECT
        user_id,
        title,
        updated_at,
        value,
        ROW_NUMBER() OVER (PARTITION BY user_id ORDER BY updated_at DESC) as rowno
    FROM silver_drj_anamnesises_epicrisis
    WHERE title = 'Treatment Type'
) x
WHERE rowno = 1
ORDER BY mpb_treatment_type_value;

In [ ]:
%%sql

SELECT DISTINCT
    value as mpb_referral_source_value
FROM silver_drj_users_restinfo
WHERE name = 'Referral Source'
ORDER BY mpb_referral_source_value;

In [ ]:
%%sql

SELECT DISTINCT
    contr_mstr_service_conformed
FROM silver_contract
WHERE contr_id LIKE 'MPB%'
ORDER BY contr_mstr_service_conformed;

mpb fin

In [ ]:
%%sql

SELECT TOP 100
    CONCAT('MPB', u.id) AS care_epi_id,
    u.id AS user_id,
    ten.id AS tenancy_id,
    ten.name AS tenancy_name,
    ten.client_type,
    rserv.service_src_name,
    rserv.service_id
FROM (SELECT * FROM silver_drj_users WHERE profile_type = 'user') u
LEFT JOIN silver_drj_tenancies ten
    ON ten.id = u.tenancy_id
LEFT JOIN silver_rdm_service rserv
    ON rserv.service_src_sys_inst_id = 'MPB001'
   AND trim(lower(rserv.service_src_name)) = trim(lower(ten.client_type))
   AND rserv.z_order_is_active = 1
ORDER BY ten.id, u.id;

In [ ]:
%%sql

SELECT
    CASE
        WHEN u.tenancy_id IS NULL THEN 'tenancy_id_null'
        WHEN ten.client_type IS NULL THEN 'client_type_null'
        WHEN rserv.service_id IS NULL THEN 'service_not_matched'
        ELSE 'service_matched'
    END as match_status,
    COUNT(*) as row_count
FROM (SELECT * FROM silver_drj_users WHERE profile_type = 'user') u
LEFT JOIN silver_drj_tenancies ten
    ON ten.id = u.tenancy_id
LEFT JOIN silver_rdm_service rserv
    ON rserv.service_src_sys_inst_id = 'MPB001'
   AND trim(lower(rserv.service_src_name)) = trim(lower(ten.client_type))
   AND rserv.z_order_is_active = 1
GROUP BY
    CASE
        WHEN u.tenancy_id IS NULL THEN 'tenancy_id_null'
        WHEN ten.client_type IS NULL THEN 'client_type_null'
        WHEN rserv.service_id IS NULL THEN 'service_not_matched'
        ELSE 'service_matched'
    END
ORDER BY row_count DESC;

In [ ]:
LEFT JOIN
    silver_rdm_service rserv
        ON rserv.service_src_sys_inst_id = 'MPB001'
       AND trim(lower(rserv.service_src_name)) = trim(lower(ten.client_type))
       AND rserv.z_order_is_active = 1

validation


In [ ]:
%%sql
SELECT
    care_epi_service_id,
    COUNT(*) as cnt
FROM silver_care_episode
WHERE z_src_system_id = 'MPB'
GROUP BY care_epi_service_id
ORDER BY cnt DESC;

In [ ]:
%%sql
SELECT TOP 50
    care_epi_id,
    care_epi_service_id,
    z_src_system_id,
    z_src_system_instance
FROM silver_care_episode
WHERE z_src_system_id = 'MPB';